In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections
import matplotlib.pyplot as plt
# %matplotlib inline

import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

In [ ]:
df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTimeGeneralInquirer.pkl', compression=None)
df_sentiment = df_sentiment.loc[df_sentiment["Date"]>"2020-05-31"]
df_sentiment["Date"]= pd.to_datetime(df_sentiment["Date"])

In [ ]:
aa = df_sentiment.loc["2020-10-01":"2020-10-25"]


In [ ]:
aa

In [ ]:
bb = aa.loc[aa["Company"]=="LIN"]

In [ ]:
print(bb.Articles.values)

In [ ]:
df_sentiment

In [ ]:
df[["VIX","Sentiment"]].plot.scatter(x = 'Sentiment', y = 'VIX', c = 'red');

In [ ]:
df_sentiment["Title"].values[4]

In [ ]:
df_sentiment.columns

# TITLE

In [ ]:
import unicodedata
dividend=0
partner=0
earnings=0
acquisition=0
public_offering=0
notes_offering=0
alliance=0
merge=0
appointment=0
ceo_compensation=0
government=0
new_brand=0
sales_down=0
covid19=0
legal_action=0
sales=0
agreement=0
undefined=0
rows_list = []

for date, row in df_sentiment.T.iteritems():
    sentence = unicodedata.normalize('NFKD', row['Title'])
    # print(date)
    if ("DIVID" in sentence):
        Type = "DIVIDEND"
        dividend+=1
    elif ("EARNING" in sentence):
        Type = "EARNING"
        earnings+=1
    elif (" EPS " in sentence):
        Type = "EARNING"
        earnings+=1    
    elif ("ACQUI" in sentence):
        Type="ACQUI"
        acquisition+=1
    elif ("PUBLIC OFFERING" in sentence):
        Type = "PUBLIC OFFERING"
        public_offering+=1
    elif ("AGREEMENT" in sentence):
        Type="AGREEMENT"
        agreement+=1
    elif ("NOTES OFFERING" in sentence):
        Type = "NOTES OFFERING"
        notes_offering+=1
    elif ("ALLIANCE" in sentence):
        Type = "ALLIANCE"
        alliance+=1
    elif ("TEAM UP" in sentence):
        Type="TEAM UP"
        alliance+=1
    elif ("MERGE" in sentence):
        Type = "MERGE"
        merge+=1
    elif ("PARTNER" in sentence):
        Type = "PARTNER"
        partner+=1
    elif ("APPOINT" in sentence):
        Type = "APPOINTMENT"
        appointment+=1
    elif ("TOTAL COMPENSATION" in sentence):
        Type = "CEO COMPENSATION"
        ceo_compensation+=1
    elif ("GOVERNMENT" in sentence):
        Type="GOVERNMENT"
        government+=1
    # elif ("SENIOR NOTES" in sentence):
    #     sales_notes+=1
    elif ("SALES" in sentence):
        Type="SALES"
        sales+=1
    # elif("COVID-19" in sentence):
    #     Type="COVID-19"
    #     covid19+=1
    #     print(sentence)
    #     print("--------------")
    # # elif("LEGAL ACTION" in sentence):
    # #     legal_action+=1
    
    else:
        Type="UNDEFINED"
        undefined+=1

        # print(sentence)
        # print("--------------")
    #     ['Time', 'Date', 'Hour', 'Negative', 'Positive', 'Sentiment', 'Polarity',
    #    'Subjectivity', 'PolarityB', 'Articles', 'Title', 'Text', 'Company'
    new_row = {"Time":row["Time"], "Date":row["Date"], "Hour": row["Hour"], "Negative":row["Negative"], "Positive":row["Positive"], "Sentiment":row["Sentiment"], "Polarity": row["Polarity"], "Subjectivity":row["Subjectivity"], "PolarityB":row["PolarityB"],"Articles":row["Articles"], "Title":row["Title"], "Text":row["Text"], "Company":row["Company"], "Type":Type}
    rows_list.append(new_row)
df = pd.DataFrame(data=rows_list, columns=["Time","Date", "Hour","Negative", "Positive", "Sentiment", "Polarity", "Subjectivity", "PolarityB","Articles","Title","Text", "Company", "Type"])
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTimeTypeGI.pkl")


In [ ]:
agreement

In [ ]:
notes_offering

In [ ]:
sales

In [ ]:
legal_action

In [ ]:
covid19

In [ ]:
merge

In [ ]:
acquisition

In [ ]:
dividend

In [ ]:
partner

In [ ]:
earnings

In [ ]:
public_offering

In [ ]:
alliance

In [ ]:
appointment

In [ ]:
ceo_compensation

In [ ]:
undefined

In [ ]:
government

In [ ]:
df

# VIX

In [ ]:
df_vix=pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl", compression=None)

df_vix.drop(columns={"open","high","low","close","volume","ticker"},inplace=True)
df_vix.rename(columns={"adjclose":"VIX"},inplace=True)
df_vix.index = pd.to_datetime(df_vix.index)
df_vix["Date"]= pd.to_datetime(df_vix.index)
df_vix = df_vix.loc[df_vix["Date"]>"2020-05-31"]

df_vix["VIX(t-1)"] = df_vix["VIX"].shift(periods=1)
df_vix["VIX(t-1)"].ffill(inplace=True)

df_vix["VIX_ln_return"]=np.log(df_vix["VIX"]/df_vix["VIX(t-1)"])
df_vix["VIX_ln_return"].ffill(inplace=True)
df = df_vix.merge(df,how="left",right_on="Date", left_on="Date")
df.loc[df["Sentiment"]==0]

In [ ]:
# df = df.loc[df["Type"]!="UNDEFINED"]

In [ ]:
sns.set(rc={'figure.figsize':(15,15)})

g =sns.scatterplot(x="Sentiment", y="VIX",
              hue="Type",
              data=df);
g.set(xscale="log");


In [ ]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['Year'] = [d.year for d in df.Date]
df['Month'] = [d.strftime('%b') for d in df.Date]
df['Day of Week'] = [d.day_name() for d in df.Date]
years = df['Year'].unique()
df['Month-Year'] = [d.strftime('%b') + str(d.year) for d in df.Date]
df['Quarter'] = pd.PeriodIndex(df.Date, freq='Q')

In [ ]:
df = df.loc[df["Date"]<"2021-12-21"]

In [ ]:
df["Sentiment_ln_return"]=np.log(df["Sentiment"].shift(periods=1)/df["Sentiment"].shift(periods=2))
df["VIX_ln_return"]=np.log(df["VIX"].shift(periods=1)/df["VIX"].shift(periods=2))

mx = df.loc[df['Sentiment_ln_return'] != np.inf, 'Sentiment_ln_return'].max()
df['Sentiment_ln_return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Sentiment_ln_return'] != -np.inf, 'Sentiment_ln_return'].min()
df['Sentiment_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['VIX_ln_return'] != np.inf, 'VIX_ln_return'].max()
df['VIX_ln_return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['VIX_ln_return'] != -np.inf, 'VIX_ln_return'].min()
df['VIX_ln_return'].replace(-np.inf,mn,inplace=True)

In [ ]:
# Draw Plot
fig, axes = plt.subplots(4, 2, figsize=(35,24), dpi= 80)
fig.tight_layout(pad=3.0)

axes[0,0].set_title('Sentiment', fontsize=24)
sns.violinplot(x='Year', y='Sentiment', data=df, ax=axes[0,0])
axes[0,1].set_title('VIX', fontsize=24)
sns.violinplot(x='Year', y='VIX', data=df, ax=axes[0,1])

axes[1,0].set_title('Sentiment', fontsize=24)
sns.violinplot(x='Day of Week', y='Sentiment', data=df,hue='Year', ax=axes[1,0], split=True)
axes[1,1].set_title('VIX', fontsize=24)
sns.violinplot(x='Day of Week', y='VIX', data=df,hue='Year', ax=axes[1,1], split=True)

axes[2,0].set_title('Sentiment Ln Return', fontsize=24)
sns.violinplot(x='Year', y='Sentiment_ln_return', data=df, ax=axes[2,0])
axes[2,1].set_title('VIX Ln Return', fontsize=24)
sns.violinplot(x='Year', y='VIX_ln_return', data=df, ax=axes[2,1])

axes[3,0].set_title('Sentiment Ln Return', fontsize=24)
sns.violinplot(x='Day of Week', y='Sentiment_ln_return', data=df,hue='Year', ax=axes[3,0], split=True)
axes[3,1].set_title('VIX Ln Return', fontsize=24)
sns.violinplot(x='Day of Week', y='VIX_ln_return', data=df,hue='Year', ax=axes[3,1], split=True,fontsize=24)


plt.savefig("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/images_press_release/GeneralInquirer/Violin/Time_sentiments.png", bbox_inches='tight')
plt.show()

# TEXT

In [ ]:
import unicodedata
dividend=0
partner=0
earnings=0
acquisition=0
public_offering=0
notes_offering=0
alliance=0
merge=0
appointment=0
ceo_compensation=0
government=0
new_brand=0
sales_down=0
covid19=0
legal_action=0
sales=0
agreement=0
undefined=0
rows_list = []

for date, row in df_sentiment.T.iteritems():
    sentence = unicodedata.normalize('NFKD', row['Articles'])
    # print(date)
    if ("DIVID" in sentence):
        Type = "DIVIDEND"
        dividend+=1        
        print(row['Title'])
        print("--------------")
    elif ("EARNING" in sentence):
        Type = "EARNING"
        earnings+=1
    elif (" EPS " in sentence):
        Type = "EARNING"
        earnings+=1    
    elif ("ACQUI" in sentence):
        Type="ACQUI"
        acquisition+=1
    elif ("PUBLIC OFFERING" in sentence):
        Type = "PUBLIC OFFERING"
        public_offering+=1
    elif ("AGREEMENT" in sentence):
        Type="AGREEMENT"
        agreement+=1
    elif ("NOTES OFFERING" in sentence):
        Type = "NOTES OFFERING"
        notes_offering+=1
    elif ("ALLIANCE" in sentence):
        Type = "ALLIANCE"
        alliance+=1
    elif ("TEAM UP" in sentence):
        Type="TEAM UP"
        alliance+=1
    elif ("MERGE" in sentence):
        Type = "MERGE"
        merge+=1
    elif ("PARTNER" in sentence):
        Type = "PARTNER"
        partner+=1
    elif ("APPOINT" in sentence):
        Type = "APPOINTMENT"
        appointment+=1
    elif ("TOTAL COMPENSATION" in sentence):
        Type = "CEO COMPENSATION"
        ceo_compensation+=1
    elif ("GOVERNMENT" in sentence):
        Type="GOVERNMENT"
        government+=1
    # elif ("SENIOR NOTES" in sentence):
    #     sales_notes+=1
    elif ("SALES" in sentence):
        Type="SALES"
        sales+=1
    # elif("COVID-19" in sentence):
    #     Type="COVID-19"
    #     covid19+=1
    #     print(sentence)
    #     print("--------------")
    # # elif("LEGAL ACTION" in sentence):
    # #     legal_action+=1
    
    else:
        Type="UNDEFINED"
        undefined+=1

        # print(sentence)
        # print("--------------")
    #     ['Time', 'Date', 'Hour', 'Negative', 'Positive', 'Sentiment', 'Polarity',
    #    'Subjectivity', 'PolarityB', 'Articles', 'Title', 'Text', 'Company'
    new_row = {"Time":row["Time"], "Date":row["Date"], "Hour": row["Hour"], "Negative":row["Negative"], "Positive":row["Positive"], "Sentiment":row["Sentiment"], "Polarity": row["Polarity"], "Subjectivity":row["Subjectivity"], "PolarityB":row["PolarityB"],"Articles":row["Articles"], "Title":row["Title"], "Text":row["Text"], "Company":row["Company"], "Type":Type}
    rows_list.append(new_row)
df = pd.DataFrame(data=rows_list, columns=["Time","Date", "Hour","Negative", "Positive", "Sentiment", "Polarity", "Subjectivity", "PolarityB","Articles","Title","Text", "Company", "Type"])
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTimeTypeGI.pkl")


In [ ]:
dividend

In [ ]:
earnings

In [ ]:
acquisition

In [ ]:
agreement

In [ ]:
notes_offering

In [ ]:
alliance

In [ ]:
merge

In [ ]:
partner

In [ ]:
appointment

In [ ]:
undefined